In [ ]:
!git clone https://github.com/JonathonLuiten/TrackEval.git

Cloning into 'TrackEval'...
remote: Enumerating objects: 924, done.
remote: Counting objects: 100% (378/378), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 924 (delta 344), reused 319 (delta 319), pack-reused 546 (from 1)
Receiving objects: 100% (924/924), 393.80 KiB | 15.75 MiB/s, done.
Resolving deltas: 100% (619/619), done.


In [ ]:
!cd TrackEval

/content/TrackEval


In [ ]:
import os

# Giả sử bạn đã biết tổng số frame lớn nhất trong 3 video của mình
max_frames = 3001  # Thay bằng số thực tế của bạn

seq_info_content = f"""[Sequence]
name=merged
imDir=im1
frameRate=10
seqLength={max_frames}
imWidth=5760
imHeight=1080
imExt=.jpg
"""

# Tạo folder nếu chưa có
os.makedirs('data/merged', exist_ok=True)

# Ghi file
with open('data/merged/seqinfo.ini', 'w') as f:
    f.write(seq_info_content)

print("Đã tạo file seqinfo.ini thành công!")

Đã tạo file seqinfo.ini thành công!


In [ ]:
import os
import pandas as pd
import sys

In [ ]:
# CẤU HÌNH GỘP FILE
CAM_WIDTH_OFFSET = 1920

def process_file(file_path, cam_index, is_gt=False):
    try:
        df = pd.read_csv(file_path, header=None)
    except pd.errors.EmptyDataError:
        return []

    df[2] = df[2] + (cam_index * CAM_WIDTH_OFFSET)
    df[7] = 1

    return df.values.tolist()

#THỰC HIỆN GỘP
gt_files = ['/content/TrackEval/data/gt/mtmc/seq1/cam1.txt', '/content/TrackEval/data/gt/mtmc/seq1/cam2.txt', '/content/TrackEval/data/gt/mtmc/seq1/cam3.txt']
pred_files = ['/content/TrackEval/data/trackers/my_track/seq1/cam1.txt', '/content/TrackEval/data/trackers/my_track/seq1/cam2.txt', '/content/TrackEval/data/trackers/my_track/seq1/cam3.txt']

all_gt_data = []
all_pred_data = []

for i in range(3):
    all_gt_data.extend(process_file(gt_files[i], i, is_gt=True))
    all_pred_data.extend(process_file(pred_files[i], i, is_gt=False))

# Lưu ra file gộp tạm thời
os.makedirs('data/merged/gt', exist_ok=True)
os.makedirs('data/merged/trackers/my_track', exist_ok=True)

# Ghi file theo format chuẩn MOT (dấu phẩy ngăn cách)
pd.DataFrame(all_gt_data).sort_values(by=[0, 1]).to_csv('data/merged/gt/gt.txt', header=False, index=False)
pd.DataFrame(all_pred_data).sort_values(by=[0, 1]).to_csv('data/merged/trackers/my_track/merged.txt', header=False, index=False)


In [ ]:
import pandas as pd

def clean_tracking_file(file_path):
    # Đọc file (format MOT: frame, id, x, y, w, h, conf, class, ...)
    df = pd.read_csv(file_path, header=None)

    # Tính diện tích để ưu tiên giữ lại Box lớn hơn (thường là cam nhìn rõ hơn)
    df['area'] = df[4] * df[5]
    df = df.sort_values(by=[0, 1, 'area'], ascending=[True, True, False])

    # Xóa trùng: Trong 1 Frame (cột 0), 1 ID (cột 1) chỉ được xuất hiện 1 lần
    df_cleaned = df.drop_duplicates(subset=[0, 1], keep='first')

    # Bỏ cột area tạm thời và lưu lại
    df_cleaned.drop(columns=['area']).to_csv(file_path, header=False, index=False)
    print(f"Đã dọn dẹp: {file_path}")

# Áp dụng cho cả 2 file đã gộp
clean_tracking_file('data/merged/gt/gt.txt')
clean_tracking_file('data/merged/trackers/my_track/merged.txt')

Đã dọn dẹp: data/merged/gt/gt.txt
Đã dọn dẹp: data/merged/trackers/my_track/merged.txt


In [ ]:
import numpy as np
# Ghi đè np.float để tương thích với code cũ của TrackEval
np.float = float
np.int = int
np.bool = bool

In [ ]:
# --- BƯỚC 3: CHẠY TRACKEVAL ---
sys.path.insert(0, os.path.abspath("TrackEval")) # Trỏ tới folder TrackEval
from trackeval.eval import Evaluator
from trackeval.datasets import MotChallenge2DBox
from trackeval.metrics import HOTA, Identity

eval_config = {'USE_PARALLEL': False, 'NUM_PARALLEL_CORES': 1, 'PRINT_RESULTS': True, 'PRINT_CONFIG': False}
evaluator = Evaluator(eval_config)

dataset_config = {
    'GT_FOLDER': 'data/merged/gt',
    'TRACKERS_FOLDER': 'data/merged/trackers',
    'BENCHMARK': 'MOT17',
    'GT_LOC_FORMAT': '{gt_folder}/gt.txt',
    'TRACKER_LOC_FORMAT': '{trackers_folder}/{tracker}/{seq}.txt',
    'CLASSES_TO_EVAL': ['pedestrian'],
    'SPLIT_TO_EVAL': 'train',
    'TRACKERS_TO_EVAL': ['my_track'],
    'SEQ_INFO': {'merged': 3001},
    'SKIP_SPLIT_FOL': True,
    'TRACKER_SUB_FOLDER': '',
    'DO_PREPROC': False
}

# Run
dataset = MotChallenge2DBox(dataset_config)
metrics_list = [HOTA(), Identity()]
evaluator.evaluate([dataset], metrics_list)


MotChallenge2DBox Config:
GT_FOLDER            : data/merged/gt                
TRACKERS_FOLDER      : data/merged/trackers          
BENCHMARK            : MOT17                         
GT_LOC_FORMAT        : {gt_folder}/gt.txt            
TRACKER_LOC_FORMAT   : {trackers_folder}/{tracker}/{seq}.txt
CLASSES_TO_EVAL      : ['pedestrian']                
SPLIT_TO_EVAL        : train                         
TRACKERS_TO_EVAL     : ['my_track']                  
SEQ_INFO             : {'merged': 3001}              
SKIP_SPLIT_FOL       : True                          
TRACKER_SUB_FOLDER   :                               
DO_PREPROC           : False                         
OUTPUT_FOLDER        : None                          
INPUT_AS_ZIP         : False                         
PRINT_CONFIG         : True                          
OUTPUT_SUB_FOLDER    :                               
TRACKER_DISPLAY_NAMES : None                          
SEQMAP_FOLDER        : None                    

({'MotChallenge2DBox': {'my_track': {'merged': {'pedestrian': {'HOTA': {'HOTA': array([0.43853412, 0.43797785, 0.43749084, 0.43664132, 0.43647391,
              0.43595078, 0.43528191, 0.43448991, 0.43301804, 0.42715286,
              0.41207504, 0.38583635, 0.34715377, 0.30228795, 0.24302364,
              0.17085623, 0.09680755, 0.03646346, 0.00379313]),
       'DetA': array([0.47886171, 0.47707116, 0.47582035, 0.47470535, 0.474438  ,
              0.47359203, 0.47252482, 0.4712817 , 0.46911127, 0.46154307,
              0.44308739, 0.40969494, 0.35942723, 0.29792271, 0.22227232,
              0.14069988, 0.06816391, 0.01900267, 0.00160099]),
       'AssA': array([0.40160273, 0.40208802, 0.40224895, 0.40162943, 0.40154767,
              0.40130128, 0.40097437, 0.40057036, 0.39970182, 0.39532511,
              0.3832333 , 0.36336717, 0.33529941, 0.30671716, 0.26571231,
              0.20747604, 0.13748773, 0.06996827, 0.00898687]),
       'DetRe': array([0.64723713, 0.64559866, 0.6444

<Figure size 640x480 with 0 Axes>